我们将线性回归模型描述为一个神经网络。 需要注意的是，该图只显示连接模式，即只显示每个输入如何连接到输出，隐去了权重和偏置的值。

输入为x1,x2...xd， 因此输入层中的输入数（或称为特征维度，feature dimensionality）为d。 网络的输出为o1，因此输出层中的输出数是1。 需要注意的是，输入值都是已经给定的，并且只有一个计算神经元。 由于模型重点在发生计算的地方，所以通常我们在计算层数时不考虑输入层。 也就是说， 神经网络的层数为1。 我们可以将线性回归模型视为仅由单个人工神经元组成的神经网络，或称为单层神经网络。

线性回归中每个输入都与每个输出相连， 我们将这种变换（ 图3.1.2中的输出层） 称为全连接层（fully-connected layer）或称为稠密层（dense layer）

实现一个线性回归，使用小批量随机数据，均方误差损失函数，线性模型

In [ ]:
import torch
def synthetic_data(w, b, num_examples):  #@save
    """生成y=Xw+b+噪声"""
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))


# 真实权重 和 偏置
true_w = torch.tensor([2, -3.4])
true_b = 4.2
# 生成1000条数据
features, labels = synthetic_data(true_w, true_b, 1000)
print(features[0],labels[0],len(features),len(labels))

tensor([-0.0574,  1.8246]) tensor([-2.1310]) 1000 1000


In [6]:
import random
import torch
def data_iter(batch_size, features, labels):
    num_examples = len(features)
    indices = list(range(num_examples))
    # 这些样本是随机读取的，没有特定的顺序
    random.shuffle(indices)
    for i in range(0, num_examples, batch_size):
        batch_indices = torch.tensor(
            indices[i: min(i + batch_size, num_examples)])
        yield features[batch_indices], labels[batch_indices]

In [ ]:
# 随机输出批量数据
for feature,label in data_iter(10,features,labels):
    print(feature,label)
    break

tensor([[-0.2961, -0.6672],
        [ 0.7999, -1.5916],
        [ 0.0634, -0.9191],
        [-0.8973,  2.0343],
        [-0.4014,  0.1327],
        [-0.1629,  1.0996],
        [-0.9607, -0.7427],
        [-1.6295,  1.0942],
        [ 1.2153, -0.9621],
        [ 0.2474, -0.3557]]) tensor([[ 5.8672],
        [11.2211],
        [ 7.4635],
        [-4.5239],
        [ 2.9477],
        [ 0.1397],
        [ 4.8064],
        [-2.7842],
        [ 9.9090],
        [ 5.9087]])


In [9]:
import torch
w = torch.normal(0, 0.01, size=(2,1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

def linreg(X, w, b):  #@save
    """线性回归模型"""
    return torch.matmul(X, w) + b

def squared_loss(y_hat, y):  #@save
    """均方损失"""
    return (y_hat - y.reshape(y_hat.shape)) ** 2 / 2

def sgd(params, lr, batch_size):  #@save
    """小批量随机梯度下降"""
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

In [ ]:
lr = 0.03
num_epochs = 10

batch_size=10
for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, labels):
        l =  squared_loss(linreg(X, w, b), y)  # X和y的小批量损失
        # 因为l形状是(batch_size,1)，而不是一个标量。l中的所有元素被加到一起，
        # 并以此计算关于[w,b]的梯度
        l.sum().backward()
        sgd([w, b], lr, batch_size)  # 使用参数的梯度更新参数
    with torch.no_grad():
        train_l =  squared_loss(linreg(features, w, b), labels)
        print(f'epoch {epoch + 1}, loss {float(train_l.mean()):f}')
print(w,b)

epoch 1, loss 0.000050
epoch 2, loss 0.000050
epoch 3, loss 0.000050
epoch 4, loss 0.000050
epoch 5, loss 0.000050
epoch 6, loss 0.000050
epoch 7, loss 0.000050
epoch 8, loss 0.000050
epoch 9, loss 0.000050
epoch 10, loss 0.000050
tensor([[ 2.0001],
        [-3.4000]], requires_grad=True) tensor([4.2006], requires_grad=True)


写入llm_kernel